#Google ADK + Ollama — Agent Server

This notebook:
1. Installs dependencies (Ollama, Google ADK, FastAPI, etc.)
2. Starts the Ollama server as a **non-blocking subprocess**
3. Defines a single Google ADK agent backed by your local Ollama model
4. Exposes `/run` and `/run_sse` HTTP endpoints via FastAPI
5. Tunnels the server publicly (via `ngrok` or `cloudflared`) so you can hit it from your device

> **Note:** Run cells in order. Steps 1-3 are one-time setup.

## Step 1 — Install Dependencies

In [ ]:
# Core packages
!pip -q install google-adk fastapi uvicorn httpx pyngrok sse-starlette

# zstd (sometimes needed for certain model formats)
!pip -q install zstd
!sudo apt-get install -y zstd 2>/dev/null | tail -1

print("✅ Python packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 44.3 MB/s eta 0:00:00
Processing triggers for man-db (2.10.2-1) ...
✅ Python packages installed


In [ ]:
!pip install google-adk[extensions]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.9/154.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.

In [ ]:
# Install Ollama binary
import shutil
if shutil.which("ollama") is None:
    print("Installing Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("✅ Ollama already installed:", shutil.which("ollama"))

Installing Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Step 2 — Start Ollama Server (Non-Blocking Subprocess)

In [ ]:
import subprocess
import time
import requests
import os

OLLAMA_BASE_URL = "http://localhost:11434"

def is_ollama_running():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if is_ollama_running():
    print("✅ Ollama server already running")
    ollama_proc = None
else:
    print("Starting Ollama server as background subprocess...")
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # Wait until the server is responsive (up to 15 s)
    for i in range(15):
        time.sleep(1)
        if is_ollama_running():
            print(f"✅ Ollama server ready (PID {ollama_proc.pid}) after {i+1}s")
            break
    else:
        print("⚠️  Ollama did not respond in 15 s — check manually")

Starting Ollama server as background subprocess...
✅ Ollama server ready (PID 6561) after 1s


## Step 3 — Pull the Model

Change `MODEL_NAME` to whichever Ollama model tag you want.

In [ ]:
# ── Configure the model you want to use ──────────────────────────────────────
# MODEL_NAME = "orchestrator-gemma"   # your custom modelfile tag
MODEL_NAME = "gemma4:latest"      # or use a stock model
# ─────────────────────────────────────────────────────────────────────────────

import json

def model_exists(model_name: str) -> bool:
    """Return True if the model is already in the local Ollama library."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        return any(model_name in m for m in models)
    except Exception:
        return False

if model_exists(MODEL_NAME):
    print(f"✅ Model '{MODEL_NAME}' already available")
else:
    print(f"Pulling '{MODEL_NAME}' — this may take a while...")
    !ollama pull gemma4:latest
    print(f"✅ '{MODEL_NAME}' ready")

Pulling 'gemma4:latest' — this may take a while...

✅ 'gemma4:latest' ready


## Step 4 — (Optional) Create a Custom Modelfile

Skip this cell if you are using a stock model.

In [ ]:
# ── Paste / edit your Modelfile here ─────────────────────────────────────────
MODELFILE_CONTENT = '''
FROM gemma4

SYSTEM """
You are a helpful assistant.
"""
'''
# ─────────────────────────────────────────────────────────────────────────────

CUSTOM_MODEL_TAG = "orchestrator-gemma"

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(MODELFILE_CONTENT.strip())
print("Wrote Modelfile")

!ollama create {CUSTOM_MODEL_TAG} -f Modelfile
!ollama ls

Wrote Modelfile

NAME                         ID              SIZE      MODIFIED               
gemma4:latest                c6eb396dbd59    9.6 GB    Less than a second ago    
orchestrator-gemma:latest    2e8868b83cb6    9.6 GB    Less than a second ago    


In [ ]:
!ollama list

NAME                         ID              SIZE      MODIFIED               
gemma4:latest                c6eb396dbd59    9.6 GB    Less than a second ago    
orchestrator-gemma:latest    2e8868b83cb6    9.6 GB    Less than a second ago    


## Step 5 — Define the Google ADK Agent

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

# LiteLLM routes to Ollama's OpenAI-compatible endpoint
ollama_model = LiteLlm(
    model=f"ollama/{MODEL_NAME}",
    api_base=OLLAMA_BASE_URL,
)

# ── Single agent definition ──────────────────────────────────────────────────
agent = LlmAgent(
    name="orchestrator_agent",
    model=ollama_model,
    description="Orchestrator agent backed by a local Ollama model.",
    instruction="You are a helpful orchestrator assistant. Answer questions thoroughly.",
    # Add tools here if needed, e.g.:  tools=[my_tool],
)

print(f"✅ ADK agent '{agent.name}' created using model '{MODEL_NAME}'")

✅ ADK agent 'orchestrator_agent' created using model 'gemma4:latest'


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## Step 6 — Build the FastAPI Server

Exposes two endpoints:

| Endpoint | Method | Description |
|---|---|---|
| `/run` | POST | Full response JSON |
| `/run_sse` | POST | Server-Sent Events (streaming) |

In [ ]:
import os
import uvicorn
from fastapi.responses import JSONResponse
from google.adk.cli.fast_api import get_fast_api_app


In [ ]:
# ── 1. Write the agent package to disk so ADK can discover it ────────────────
AGENT_PACKAGE = "orchestrator_agent"  # becomes the app_name in API requests
os.makedirs(AGENT_PACKAGE, exist_ok=True)

agent_init = f"""
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

ollama_model = LiteLlm(
    model="ollama/{MODEL_NAME}",
    api_base="{OLLAMA_BASE_URL}",
)

root_agent = LlmAgent(
    name="orchestrator_agent",
    model=ollama_model,
    description="Orchestrator agent backed by a local Ollama model.",
    instruction="You are a helpful orchestrator assistant. Answer questions thoroughly.",
    # Add tools here if needed, e.g.: tools=[my_tool],
)
"""

with open(f"{AGENT_PACKAGE}/__init__.py", "w") as f:
    f.write(agent_init.strip())
print(f"✅ Agent package written to ./{AGENT_PACKAGE}/__init__.py")


✅ Agent package written to ./orchestrator_agent/__init__.py


In [ ]:

# ── 2. Build the ADK FastAPI app ─────────────────────────────────────────────
AGENT_DIR = os.getcwd()  # parent directory containing the agent package folder

app = get_fast_api_app(
    agents_dir=AGENT_DIR,
    allow_origins=["*"],
    web=False,   # set True to also serve the ADK dev UI at /dev-ui
)

# ── 3. Add a /health endpoint on top ─────────────────────────────────────────
# @app.get("/health")
# async def health():
#     return {"status": "ok", "model": MODEL_NAME, "app_name": AGENT_PACKAGE}

print("✅ ADK FastAPI app ready — proceed to Step 7 to start the server")
print(f"   Use app_name='{AGENT_PACKAGE}' in all API requests")

✅ ADK FastAPI app ready — proceed to Step 7 to start the server
   Use app_name='orchestrator_agent' in all API requests


/usr/local/lib/python3.12/dist-packages/google/adk/cli/fast_api.py:198: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [ ]:
# Re-run this check before Step 7
import requests, subprocess, time

OLLAMA_BASE_URL = "http://localhost:11434"

def is_ollama_running():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if not is_ollama_running():
    print("Restarting Ollama...")
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for i in range(15):
        time.sleep(1)
        if is_ollama_running():
            print(f"✅ Ready after {i+1}s")
            break
    else:
        print("⚠️ Still not responding")
else:
    print("✅ Already running")

✅ Already running


## Step 7 — Start the Server

Uvicorn runs in a background thread so the rest of the notebook stays interactive.

In [ ]:
import threading

SERVER_HOST = "0.0.0.0"
SERVER_PORT = 8000

def _run_server():
    uvicorn.run(app, host=SERVER_HOST, port=SERVER_PORT, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Give uvicorn a moment to bind
time.sleep(2)
print(f"✅ Server running on http://{SERVER_HOST}:{SERVER_PORT}")
print(f"   POST /run      — full JSON response")
print(f"   POST /run_sse  — streaming SSE response")
print(f"   GET  /health   — liveness check")

✅ Server running on http://0.0.0.0:8000
   POST /run      — full JSON response
   POST /run_sse  — streaming SSE response
   GET  /health   — liveness check


## Step 8 — Expose Publicly via Ngrok

This gives you a public HTTPS URL you can call from your phone or any device.

Get a free auth-token from https://dashboard.ngrok.com and paste it below.

In [ ]:
from pyngrok import ngrok, conf

# ── Paste your ngrok auth token here ─────────────────────────────────────────
NGROK_AUTH_TOKEN = "399VSD7MO0sWalwjnkmCi5DXPu2_2rjWEzbbPU3UbKhdMVowN"
# ─────────────────────────────────────────────────────────────────────────────

if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    ngrok.kill()  # kill any existing tunnels
    tunnel = ngrok.connect(SERVER_PORT, "http")
    PUBLIC_URL = tunnel.public_url
    print(f"🌐 Public URL: {PUBLIC_URL}")
    print(f"   /run     → {PUBLIC_URL}/run")
    print(f"   /run_sse → {PUBLIC_URL}/run_sse")
    print(f"   /health  → {PUBLIC_URL}/health")
else:
    print("⚠️  Set NGROK_AUTH_TOKEN above, then re-run this cell.")
    print(f"   Local URL (Colab only): http://localhost:{SERVER_PORT}")

🌐 Public URL: https://unseclusive-katlyn-weakheartedly.ngrok-free.dev
   /run     → https://unseclusive-katlyn-weakheartedly.ngrok-free.dev/run
   /run_sse → https://unseclusive-katlyn-weakheartedly.ngrok-free.dev/run_sse
   /health  → https://unseclusive-katlyn-weakheartedly.ngrok-free.dev/health


Step 9 — Test Locally Inside the Notebook

In [ ]:
import httpx

LOCAL = f"http://localhost:{SERVER_PORT}"

# ── Health check ──────────────────────────────────────────────────────────────
r = httpx.get(f"{LOCAL}/health")
print("Health:", r.json())

# ── Create a session (required before /run or /run_sse) ───────────────────────
USER_ID = "test-user"
SESSION_ID = "test-session-002"

r = httpx.post(
    f"{LOCAL}/apps/{AGENT_PACKAGE}/users/{USER_ID}/sessions/{SESSION_ID}",
    json={},
    timeout=10
)
print("Session created:", r.status_code, r.text[:200])

Health: {'status': 'ok'}
Session created: 200 {"id":"test-session-002","appName":"orchestrator_agent","userId":"test-user","state":{},"events":[],"lastUpdateTime":1777724897.4531517}


In [ ]:
# Re-run this check before Step 7
import requests, subprocess, time

OLLAMA_BASE_URL = "http://localhost:11434"

def is_ollama_running():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if not is_ollama_running():
    print("Restarting Ollama...")
    ollama_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for i in range(15):
        time.sleep(1)
        if is_ollama_running():
            print(f"✅ Ready after {i+1}s")
            break
    else:
        print("⚠️ Still not responding")
else:
    print("✅ Already running")

✅ Already running


In [ ]:
# ── POST /run (streaming token-by-token via response events) ─────────────────
# /run returns all events at once as a JSON array — we stream the HTTP response
# body and print each token as its event chunk arrives, before the array closes.
import json as _json

payload = {
    "app_name": AGENT_PACKAGE,
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "new_message": {
        "role": "user",
        "parts": [{"text": "Hello! What can you help me with?"}]
    }
}

print("Streaming response (tokens as they arrive):")
print("-" * 50)

buffer = ""
with httpx.stream("POST", f"{LOCAL}/run", json=payload, timeout=300) as response:
    for raw_bytes in response.iter_bytes():
        buffer += raw_bytes.decode("utf-8", errors="replace")
        # /run returns a JSON array of event objects streamed incrementally.
        # Each complete event object ends with a closing brace; we scan for
        # complete {...} objects inside the array as they land.
        depth = 0
        obj_start = None
        i = 0
        consumed = 0
        while i < len(buffer):
            ch = buffer[i]
            if ch == "{":
                if depth == 0:
                    obj_start = i
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0 and obj_start is not None:
                    fragment = buffer[obj_start:i+1]
                    try:
                        event = _json.loads(fragment)
                        content = event.get("content") or {}
                        for part in content.get("parts", []):
                            token = part.get("text", "")
                            if token:
                                print(token, end="", flush=True)
                    except _json.JSONDecodeError:
                        pass
                    consumed = i + 1
                    obj_start = None
            i += 1
        buffer = buffer[consumed:]

print("\n" + "-" * 50)
print("[stream complete]")

# ─────────────────────────────────────────────────────────────────────────────
# curl equivalent (run on your device):
# ─────────────────────────────────────────────────────────────────────────────
# First create a session:
#   curl -X POST https://<ngrok-url>/apps/orchestrator_agent/users/u1/sessions/s1 \
#        -H "Content-Type: application/json" -d '{}'
#
# Then call /run:
#   curl -X POST https://<ngrok-url>/run \
#        -H "Content-Type: application/json" \
#        -d '{
#              "app_name": "orchestrator_agent",
#              "user_id": "u1",
#              "session_id": "s1",
#              "new_message": {"role": "user", "parts": [{"text": "Hello!"}]}
#            }'
# Note: /run buffers all events then returns them — no --no-buffer needed.


Streaming response (tokens as they arrive):
--------------------------------------------------

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



KeyboardInterrupt: 

In [ ]:
# ── POST /run_sse (streaming SSE — token by token) ───────────────────────────
import json as _json

SESSION_ID_2 = "test-session-002"
httpx.post(
    f"{LOCAL}/apps/orchestrator_agent/users/{USER_ID}/sessions/{SESSION_ID_2}",
    json={}, timeout=10
)

payload_sse = {
    "app_name": "orchestrator_agent",
    "user_id": USER_ID,
    "session_id": SESSION_ID_2,
    "new_message": {
        "role": "user",
        "parts": [
        {"text": "What do you see in this image?"},
        # {
        #   "inlineData": {
        #     "displayName":"my_image.png",
        #     "data": "",
        #     "mimeType": "image/jpeg",
        #   }
        # }
      ]
    },
    "streaming": True
}

print("Streaming response (tokens as they arrive):")
print("-" * 50)


with httpx.stream(
    "POST",
    f"{LOCAL}/run_sse",
    json=payload_sse,
    headers={"Accept": "text/event-stream"},
    timeout=300,
) as response:
    buffer = ""
    for raw_bytes in response.iter_bytes():
        buffer += raw_bytes.decode("utf-8", errors="replace")
        while "\n" in buffer:
            line, buffer = buffer.split("\n", 1)
            line = line.strip()
            if not line.startswith("data:"):
                continue
            raw = line[len("data:"):].strip()
            if not raw or raw == "[DONE]":
                continue
            try:
                event = _json.loads(raw)
            except _json.JSONDecodeError:
                continue

            content = event.get("content") or {}
            for part in content.get("parts", []):
                token = part.get("text", "")
                if token:
                    print(token, end="", flush=True)

print("\n" + "-" * 50)
print("[stream complete]")

Streaming response (tokens as they arrive):
--------------------------------------------------


KeyboardInterrupt: 

In [ ]:
# ── POST /run_sse — image upload + fresh session every run ──────────────────
import json as _json, uuid as _uuid, base64 as _b64
from google.colab import files as _files

# ── Step 1: Upload image via Colab file picker ────────────────────────────────
print("📁 Select an image file to upload...")
_uploaded = _files.upload()   # opens the file picker
_filename = list(_uploaded.keys())[0]
_raw_bytes = _uploaded[_filename]

# Detect mime type from extension
_ext = _filename.rsplit('.', 1)[-1].lower()
_mime_map = {'jpg': 'image/jpeg', 'jpeg': 'image/jpeg', 'png': 'image/png',
             'gif': 'image/gif', 'webp': 'image/webp', 'bmp': 'image/bmp'}
_mime = _mime_map.get(_ext, 'image/jpeg')

# Convert to clean single-line base64 (no newlines — newlines break JSON)
_img_b64 = _b64.b64encode(_raw_bytes).decode('utf-8')

print(f"✅ Loaded: {_filename}")
print(f"   Size:      {len(_raw_bytes):,} bytes")
print(f"   MIME:      {_mime}")
print(f"   Base64:    {len(_img_b64):,} chars (first 40: {_img_b64[:40]}...)")

# ── Step 2: Create a FRESH session every run ──────────────────────────────────
# This is critical — reusing a session means ADK replays cached history
# and the model sees the old image context, not the new one.
_session_id = f"img-{_uuid.uuid4().hex[:10]}"
_r = httpx.post(
    f"{LOCAL}/apps/orchestrator_agent/users/{USER_ID}/sessions/{_session_id}",
    json={}, timeout=10,
)
print(f"\n✅ Fresh session created: {_session_id} (status {_r.status_code})")

# ── Step 3: Build payload with inline image ───────────────────────────────────
_payload = {
    "app_name": "orchestrator_agent",
    "user_id": USER_ID,
    "session_id": _session_id,
    "new_message": {
        "role": "user",
        "parts": [
            {"text": "What do you see in this image? Describe it in detail."},
            {
                "inline_data": {
                    "mime_type": _mime,
                    "data": _img_b64,      # clean base64, no newlines
                }
            }
        ]
    },
    "streaming": True,
}

# Quick sanity check — serialize and back to confirm JSON is valid
_test = _json.loads(_json.dumps(_payload))
print(f"✅ Payload JSON valid — inline_data present: {'inline_data' in str(_test)}")

# ── Step 4: Stream the response ───────────────────────────────────────────────
print("\nStreaming response (tokens as they arrive):")
print("-" * 50)

_token_count = 0
with httpx.stream(
    "POST",
    f"{LOCAL}/run_sse",
    json=_payload,
    headers={"Accept": "text/event-stream"},
    timeout=300,
) as _response:
    print(f"HTTP {_response.status_code}")
    _buffer = ""
    for _chunk in _response.iter_bytes():
        _buffer += _chunk.decode("utf-8", errors="replace")
        while "\n" in _buffer:
            _line, _buffer = _buffer.split("\n", 1)
            _line = _line.strip()
            if not _line.startswith("data:"):
                continue
            _raw = _line[5:].strip()
            if not _raw or _raw == "[DONE]":
                continue
            try:
                _event = _json.loads(_raw)
            except _json.JSONDecodeError:
                continue
            for _part in (_event.get("content") or {}).get("parts", []):
                _token = _part.get("text", "")
                if _token:
                    print(_token, end="", flush=True)
                    _token_count += 1

print("\n" + "-" * 50)
print(f"[stream complete — {_token_count} text event(s) received]")
if _token_count == 0:
    print("\n⚠️  No tokens received. The model likely cannot process inline_data via ADK/LiteLLM.")
    print("   Use the /analyze-medical-report endpoint instead — it calls Ollama vision directly.")


## Calling the APIs from Your Device

Use the public URL printed in Step 8.

**`/{LOCAL}/apps/{AGENT_PACKAGE}/users/{USER_ID}/sessions/{SESSION_ID}`**
```bash
curl -k -X POST https://unseclusive-katlyn-weakheartedly.ngrok-free.dev/apps/orchestrator_agent/users/u1/sessions/s1 -H "Content-Type: application/json" -d "{}"
```

**`/run` — full response (curl)**
```bash
curl -k -X POST https://unseclusive-katlyn-weakheartedly.ngrok-free.dev/run -H "Content-Type: application/json" -d "{\"app_name\":\"orchestrator_agent\",\"user_id\":\"u1\",\"session_id\":\"s1\",\"new_message\":{\"role\":\"user\",\"parts\":[{\"text\":\"Hello!\"}]}}"
```

**`/run_sse` — streaming (curl)**
```bash
curl -k -X POST https://unseclusive-katlyn-weakheartedly.ngrok-free.dev/run_sse -H "Content-Type: application/json" -H "Accept: text/event-stream" --no-buffer -d "{\"app_name\":\"orchestrator_agent\",\"user_id\":\"u1\",\"session_id\":\"s2\",\"new_message\":{\"role\":\"user\",\"parts\":[{\"text\":\"Hello!\"}]},\"streaming\":true}"
```

**Request body fields**

| Field | Type | Required | Description |
|---|---|---|---|
| `message` | string | ✅ | The user's input text |
| `session_id` | string | optional | Reuse to keep conversation context |
| `user_id` | string | optional | Identifies the user (defaults to `default_user`) |

## Shutdown (Optional)

Run this cell to cleanly kill the Ollama server subprocess when you are done.

In [ ]:
if 'ollama_proc' in globals() and ollama_proc is not None:
    ollama_proc.terminate()
    ollama_proc.wait()
    print("✅ Ollama server stopped")
else:
    print("No subprocess to stop (was already running or not started here)")

try:
    ngrok.kill()
    print("✅ Ngrok tunnel closed")
except Exception:
    pass

✅ Ollama server stopped
✅ Ngrok tunnel closed


Test Flutter

In [ ]:
import httpx

LOCAL = "http://localhost:8000"
USER_ID = "69dfd6f13553c8894c4e8797"
SESSION_ID = "1776717656492"
AGENT_PACKAGE = "orchestrator_agent"

# 1. Create session
r = httpx.post(f"{LOCAL}/apps/{AGENT_PACKAGE}/users/{USER_ID}/sessions/{SESSION_ID}",
               json={}, timeout=10)
print("Session:", r.status_code, r.text[:200])


Session: 409 {"detail":"Session already exists: 1776717656492"}


In [ ]:

# 2. Test /run_sse with streaming=False first (simpler, rules out SSE issue)
payload = {
    "app_name": AGENT_PACKAGE,
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "new_message": {
        "role": "user",
        "parts": [{"text": "hi"}]
    },
    "streaming": False
}

print("\n--- Testing /run_sse with streaming=False ---")
try:
    with httpx.stream("POST", f"{LOCAL}/run_sse",
                      json=payload,
                      headers={"Accept": "text/event-stream"},
                      timeout=60) as r:
        print("Status:", r.status_code)
        for line in r.iter_lines():
            if line.startswith("data:"):
                print("EVENT:", line[:300])
except Exception as e:
    print("ERROR:", e)

# 3. Test /run_sse with streaming=True
payload["streaming"] = True
print("\n--- Testing /run_sse with streaming=True ---")
try:
    with httpx.stream("POST", f"{LOCAL}/run_sse",
                      json=payload,
                      headers={"Accept": "text/event-stream"},
                      timeout=60) as r:
        print("Status:", r.status_code)
        for line in r.iter_lines():
            if line.startswith("data:"):
                print("TOKEN:", line[:200])
except Exception as e:
    print("ERROR:", e)


--- Testing /run_sse with streaming=False ---
Status: 404

--- Testing /run_sse with streaming=True ---
Status: 404
